# Orderbook Encoder - L2 Order Book MLP (XRPUSDT)

Trains a deep MLP encoder on sampled L2 order book states (Bybit, XRPUSDT).

- **Input**: 80-dim feature vector from the top-20 levels per side, layout
  `[bid_off x20, bid_qty x20, ask_off x20, ask_qty x20]` where
  `off = |price - mid| / mid` and `qty = log1p(qty)`, sampled at 1 Hz.
- **Target**: mid-price move over a 60 s horizon, 3 classes
  `DOWN(0) / FLAT(1) / UP(2)` with a +/-0.05% return threshold.
- **Model**: stack of `Linear -> LayerNorm -> GELU -> Dropout` blocks down to a
  64-dim embedding, plus a linear classification head.
- **Data**: Kaggle dataset `suenot/orderbook-samples-xrpusdt` with one `.npz`
  per day (keys: `ts` int64 ms, `features` float32 `(T, 80)`, `mid` float64).

In [ ]:
"""Configuration: data location, splits, model and training hyper-parameters."""
import torch

CFG = {
    "data": {
        "samples_dir": "/kaggle/input",
        "symbol": "XRPUSDT",
        "exchange": "bybit",
    },
    "split": {
        "train_days": [
            "2026-06-01", "2026-06-02", "2026-06-03",
            "2026-06-04", "2026-06-05", "2026-06-06",
        ],
        "val_days": ["2026-06-07", "2026-06-08"],
        "test_days": ["2026-06-09"],
    },
    "model": {
        "input_dim": 80,           # 4 * depth(20)
        "hidden_dims": [256, 128],
        "embedding_dim": 64,
        "num_classes": 3,
        "dropout": 0.1,
    },
    "training": {
        "batch_size": 256,
        "learning_rate": 1e-3,
        "weight_decay": 0.01,
        "epochs": 10,
        "early_stop_patience": 3,
        "artifacts_dir": "/kaggle/working/artifacts/orderbook_encoder",
    },
    "target": {
        "horizon_sec": 60,
        "threshold_pct": 0.05,
        "interval_sec": 1.0,
    },
}

print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")

from pathlib import Path
print("\n/kaggle/input tree:")
for f in sorted(Path("/kaggle/input").rglob("*"))[:40]:
    print(" ", f)


In [ ]:
"""Dataset over sampled order book npz files.

Each npz holds one day of 1 Hz samples with keys:
    ts:       int64 (T,)  millisecond timestamps
    features: float32 (T, 4*depth)
    mid:      float64 (T,)  mid-price

Labels predict the mid-price move over ``horizon_sec``:
    ret = mid[i+h] / mid[i] - 1
    UP(2)   if ret >  threshold_pct/100
    DOWN(0) if ret < -threshold_pct/100
    FLAT(1) otherwise

Windows never cross npz-file boundaries. A pair (i, i+h) whose timestamp gap
deviates from ``horizon_sec*1000`` by more than that amount (a hole in the data)
is dropped. The list of valid (file, i) pairs is built once in ``__init__``.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset


DOWN, FLAT, UP = 0, 1, 2


class OrderbookDataset(Dataset):
    def __init__(
        self,
        npz_paths: list[Path],
        horizon_sec: float,
        threshold_pct: float,
        interval_sec: float,
    ) -> None:
        self.horizon_sec = horizon_sec
        self.threshold_pct = threshold_pct
        self.interval_sec = interval_sec
        self.h = max(1, round(horizon_sec / interval_sec))

        self._features: list[np.ndarray] = []
        self._mid: list[np.ndarray] = []
        # (file_index, i): the row i and its partner i+h are both valid.
        self._pairs: list[tuple[int, int]] = []

        gap_ms = horizon_sec * 1000.0
        for path in npz_paths:
            with np.load(path) as data:
                ts = data["ts"].astype(np.int64)
                feats = data["features"].astype(np.float32)
                mid = data["mid"].astype(np.float64)
            file_idx = len(self._features)
            self._features.append(feats)
            self._mid.append(mid)

            n = len(ts)
            for i in range(n - self.h):
                j = i + self.h
                if abs((ts[j] - ts[i]) - gap_ms) > gap_ms:
                    continue
                self._pairs.append((file_idx, i))

    def __len__(self) -> int:
        return len(self._pairs)

    def _label(self, mid: np.ndarray, i: int, j: int) -> int:
        ret = mid[j] / mid[i] - 1.0
        thr = self.threshold_pct / 100.0
        if ret > thr:
            return UP
        if ret < -thr:
            return DOWN
        return FLAT

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        file_idx, i = self._pairs[idx]
        j = i + self.h
        feats = self._features[file_idx][i]
        label = self._label(self._mid[file_idx], i, j)
        x = torch.from_numpy(np.ascontiguousarray(feats)).to(torch.float32)
        y = torch.tensor(label, dtype=torch.int64)
        return x, y


def _split_paths(cfg: dict, days: list[str]) -> list[Path]:
    """Resolve npz paths for ``days`` by recursive glob; warn about missing days.

    The dataset zip may be extracted with or without its top-level directories
    depending on how Kaggle unpacked it, so we search the whole input tree.
    """
    root = Path(cfg["data"]["samples_dir"])
    found = {f.stem: f for f in sorted(root.rglob("*.npz"))}
    paths: list[Path] = []
    for date in days:
        if date in found:
            paths.append(found[date])
        else:
            print(f"Warning: no samples npz found for {date} under {root}")
    return paths


def build_splits(
    cfg: dict,
) -> tuple[OrderbookDataset, OrderbookDataset, OrderbookDataset]:
    """Build train/val/test datasets from date lists in ``cfg['split']``."""
    target = cfg["target"]
    split = cfg["split"]

    def make(days: list[str]) -> OrderbookDataset:
        return OrderbookDataset(
            npz_paths=_split_paths(cfg, days),
            horizon_sec=target["horizon_sec"],
            threshold_pct=target["threshold_pct"],
            interval_sec=target["interval_sec"],
        )

    return make(split["train_days"]), make(split["val_days"]), make(split["test_days"])

In [ ]:
"""Deep MLP encoder for L2 order book feature vectors.

OrderbookEncoder maps a flat feature vector to a compact embedding through a
stack of Linear -> LayerNorm -> GELU -> Dropout blocks. OrderbookClassifier
wraps the encoder with a linear head for the 3-class movement target.
"""
from __future__ import annotations

import torch
import torch.nn as nn


class OrderbookEncoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dims: list[int],
        embedding_dim: int,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        prev = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev, h),
                nn.LayerNorm(h),
                nn.GELU(),
                nn.Dropout(dropout),
            ]
            prev = h
        layers.append(nn.Linear(prev, embedding_dim))
        self.net = nn.Sequential(*layers)
        self.embedding_dim = embedding_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class OrderbookClassifier(nn.Module):
    def __init__(self, encoder: OrderbookEncoder, num_classes: int) -> None:
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(encoder.embedding_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.encoder(x))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)

In [ ]:
"""Training loop for the order book MLP classifier.

``train(cfg)`` builds the train/val/test splits, trains the classifier with
AdamW and class-weighted cross-entropy, early-stops on validation loss, and
writes artifacts to ``{artifacts_dir}/run_YYYYMMDD_HHMMSS/``:
    best.pt            best model state dict (by val loss)
    config.json        the config used for the run
    test_metrics.json  classification report + confusion matrix on the test set
"""
from __future__ import annotations

import json
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix

# OrderbookDataset / build_splits and the model classes come from the cells above.

TARGET_NAMES = ["DOWN", "FLAT", "UP"]


def compute_class_weights(labels: list[int], num_classes: int = 3) -> list[float]:
    """Weights inversely proportional to class frequency in ``labels``."""
    counts = Counter(labels)
    total = max(len(labels), 1)
    weights = []
    for c in range(num_classes):
        cnt = counts.get(c, 1)
        weights.append(total / (num_classes * cnt))
    return weights


def _auto_device() -> str:
    """Pick cuda when available, otherwise cpu."""
    return "cuda" if torch.cuda.is_available() else "cpu"


def _train_labels(ds: OrderbookDataset) -> list[int]:
    return [int(ds[i][1]) for i in range(len(ds))]


def _run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: str,
    optimizer: AdamW | None = None,
) -> tuple[float, list[int], list[int]]:
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss = 0.0
    n_batches = 0
    preds: list[int] = []
    labels: list[int] = []

    with torch.set_grad_enabled(train_mode):
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            n_batches += 1
            preds.extend(logits.argmax(dim=-1).cpu().numpy().tolist())
            labels.extend(y.cpu().numpy().tolist())

    return total_loss / max(n_batches, 1), preds, labels


def train(cfg: dict) -> dict:
    train_ds, val_ds, test_ds = build_splits(cfg)
    print(f"Datasets: train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

    tcfg = cfg["training"]
    mcfg = cfg["model"]
    device = _auto_device()
    print(f"Device: {device}")
    batch_size = tcfg["batch_size"]

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    encoder = OrderbookEncoder(
        input_dim=mcfg["input_dim"],
        hidden_dims=mcfg["hidden_dims"],
        embedding_dim=mcfg["embedding_dim"],
        dropout=mcfg["dropout"],
    )
    model = OrderbookClassifier(encoder, num_classes=mcfg["num_classes"]).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}")

    class_weights = compute_class_weights(_train_labels(train_ds), mcfg["num_classes"])
    weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = AdamW(
        model.parameters(), lr=tcfg["learning_rate"], weight_decay=tcfg["weight_decay"]
    )

    run_dir = Path(tcfg["artifacts_dir"]) / f"run_{datetime.now():%Y%m%d_%H%M%S}"
    run_dir.mkdir(parents=True, exist_ok=True)
    with open(run_dir / "config.json", "w") as f:
        json.dump(cfg, f, indent=2)

    best_val = float("inf")
    patience = 0
    for epoch in range(1, tcfg["epochs"] + 1):
        train_loss, _, _ = _run_epoch(model, train_dl, criterion, device, optimizer)
        val_loss, _, _ = _run_epoch(model, val_dl, criterion, device)
        print(f"Epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            patience = 0
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict()},
                       run_dir / "best.pt")
        else:
            patience += 1
            if patience >= tcfg["early_stop_patience"]:
                print(f"Early stopping at epoch {epoch}")
                break

    # Evaluate the best checkpoint on the test set.
    best = torch.load(run_dir / "best.pt", map_location=device, weights_only=True)
    model.load_state_dict(best["model_state_dict"])

    if len(test_ds) == 0:
        print("Warning: empty test split, skipping test metrics")
        report: dict = {}
        cm = np.zeros((3, 3), dtype=int)
    else:
        _, test_preds, test_labels = _run_epoch(model, test_dl, criterion, device)
        report = classification_report(
            test_labels, test_preds, target_names=TARGET_NAMES,
            labels=[0, 1, 2], output_dict=True, zero_division=0,
        )
        cm = confusion_matrix(test_labels, test_preds, labels=[0, 1, 2])

    metrics = {
        "report": report,
        "confusion_matrix": cm.tolist(),
        "best_val_loss": best_val,
    }
    with open(run_dir / "test_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    metrics["run_dir"] = str(run_dir)
    return metrics

In [ ]:
"""Run training and report test-set metrics."""
metrics = train(CFG)

report = metrics["report"]
if report:
    print(f"\nTest macro F1: {report['macro avg']['f1-score']:.4f}")
    print(f"Test accuracy: {report['accuracy']:.4f}")
else:
    print("\nNo test metrics (empty test split).")

print("Confusion matrix (rows = true, cols = pred; order DOWN/FLAT/UP):")
for name, row in zip(TARGET_NAMES, metrics["confusion_matrix"]):
    print(f"  {name:>4}: {row}")
print(f"Run dir: {metrics['run_dir']}")

In [ ]:
"""List produced artifact files with their sizes."""
import os

artifacts_root = Path(CFG["training"]["artifacts_dir"])
for dirpath, _dirnames, filenames in sorted(os.walk(artifacts_root)):
    for fname in sorted(filenames):
        fpath = Path(dirpath) / fname
        print(f"{fpath}  ({fpath.stat().st_size:,} bytes)")